In [ ]:
import  os
import  certifi
import  requests
from dotenv import load_dotenv

from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain_tavily import TavilySearch
from langchain_classic import hub
from langchain.agents import create_agent

In [ ]:
#loading environment variables

os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

GROQ_API_KEY = os.getenv("OPEN_AI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHER_STACK = os.getenv("WEATHER_STACK")

In [ ]:
search_tool = TavilySearch(max_results=3)

In [ ]:
@tool
def get_weather_data(city: str) -> str:
    url = (
        f"https://api.weatherstack.com/current?"
        f"api_key={WEATHER_STACK}&query={city}"
    )
    response = requests.get(url)
    data = response.json()

    if "current" not in data:
        return f" could not get weather data for {city} "

    return(
        f"city: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )

In [ ]:
result = search_tool.invoke("Give me the latest news on AI")
result

In [ ]:
# LLM

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
    api_key=GROQ_API_KEY
)

In [ ]:
response = llm.invoke("tell me a joke?")
response

In [ ]:
agent = create_agent(
    model=llm,
    tools=[search_tool],
    system_prompt="You are a helpful research assistant. Use the search tool when you need current or realtime information."
)

In [ ]:
res = agent.invoke({
    "messages":(
        "find the capital of nigeria and the weather conditions in kaduna as at 24 8 2026"
    ) })
print(res["messages"][-1].content)